# FraudShield â€” Model Training on Google Colab
**DSBDAL Mini Project â€” Fraud Detection in Banking Transactions**

## Steps:
1. Upload `creditcard.csv` when prompted
2. Run all cells (Runtime â†’ Run All)
3. Download the zip file at the end
4. Extract into your `backend/models/` folder

---
**Dataset:** [Kaggle Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

In [ ]:
# â”€â”€ Step 1: Install required packages â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
!pip install -q imbalanced-learn xgboost scikit-learn pandas numpy joblib

In [ ]:
# â”€â”€ Step 2: Upload creditcard.csv â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from google.colab import files
import os

print('Upload creditcard.csv from your machine:')
uploaded = files.upload()

csv_path = list(uploaded.keys())[0]
print(f'\nUploaded: {csv_path} ({os.path.getsize(csv_path)/1e6:.1f} MB)')

In [ ]:
# â”€â”€ Step 3: Load and inspect data â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(csv_path)

print('=' * 50)
print(f'Shape          : {df.shape}')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Duplicates     : {df.duplicated().sum()}')
print('=' * 50)

class_counts = df['Class'].value_counts()
fraud_pct = class_counts[1] / len(df) * 100
print(f'Legitimate : {class_counts[0]:,} ({100 - fraud_pct:.2f}%)')
print(f'Fraud      : {class_counts[1]:,} ({fraud_pct:.4f}%)')
print(f'Imbalance  : {class_counts[0] // class_counts[1]}:1')

In [ ]:
# â”€â”€ Step 4: Preprocessing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Scale Amount and Time
df2 = df.copy()
df2['Amount_Scaled'] = RobustScaler().fit_transform(df2[['Amount']])
df2['Time_Scaled']   = StandardScaler().fit_transform(df2[['Time']])
df2.drop(columns=['Amount', 'Time'], inplace=True)

X = df2.drop('Class', axis=1)
y = df2['Class']

# Train/test split (stratified to preserve fraud ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train fraud: {y_train.sum()} | Test fraud: {y_test.sum()}')

# Apply SMOTE only on training set
smote = SMOTE(sampling_strategy=0.1, k_neighbors=5, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'\nAfter SMOTE â€” Train: {X_train_res.shape}')
print(f'Fraud samples: {y_train_res.sum()} (was {y_train.sum()})')

In [ ]:
# â”€â”€ Step 5: Train Logistic Regression â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from sklearn.linear_model import LogisticRegression
import joblib, os

os.makedirs('models', exist_ok=True)

print('Training Logistic Regression...')
lr = LogisticRegression(
    class_weight='balanced', C=0.01,
    max_iter=1000, solver='lbfgs', random_state=42
)
lr.fit(X_train_res, y_train_res)
joblib.dump(lr, 'models/logistic_regression.pkl')
print('Saved â†’ models/logistic_regression.pkl')

In [ ]:
# â”€â”€ Step 6: Train Random Forest â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from sklearn.ensemble import RandomForestClassifier

print('Training Random Forest (this takes ~2-3 min)...')
rf = RandomForestClassifier(
    n_estimators=100, max_depth=15,
    min_samples_split=10, min_samples_leaf=5,
    class_weight='balanced', n_jobs=-1,
    random_state=42, oob_score=True
)
rf.fit(X_train_res, y_train_res)
print(f'OOB Score: {rf.oob_score_:.4f}')
joblib.dump(rf, 'models/random_forest.pkl')
print('Saved â†’ models/random_forest.pkl')

In [ ]:
# ── Step 7: Train XGBoost (saves native .json to avoid version warnings) ──
from xgboost import XGBClassifier

scale_pos = int(y_train_res.value_counts()[0] / y_train_res.value_counts()[1])
print(f"Training XGBoost (scale_pos_weight={scale_pos})...")

xgb = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos, subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr", random_state=42, n_jobs=-1
)
xgb.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# Save in native XGBoost JSON format — no version-mismatch warning on load
xgb.get_booster().save_model("models/xgboost_model.json")
print("Saved → models/xgboost_model.json  (native format, no warnings)")

# Also keep .pkl for compatibility
import joblib
joblib.dump(xgb, "models/xgboost_model.pkl")
print("Saved → models/xgboost_model.pkl   (pickle fallback)")


In [ ]:
# â”€â”€ Step 8: Train Isolation Forest â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from sklearn.ensemble import IsolationForest

print('Training Isolation Forest...')
iso = IsolationForest(
    n_estimators=200, contamination=0.001727,
    max_samples='auto', random_state=42, n_jobs=-1
)
iso.fit(X_train)   # unsupervised â€” train on original (no SMOTE)
joblib.dump(iso, 'models/isolation_forest.pkl')
print('Saved â†’ models/isolation_forest.pkl')

In [ ]:
# â”€â”€ Step 9: Evaluate all models â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score
)

def eval_model(name, model, X_test, y_test, is_iso=False):
    if is_iso:
        raw = model.predict(X_test)
        y_pred = np.where(raw == -1, 1, 0)
        y_prob = -model.score_samples(X_test)
        y_prob = (y_prob - y_prob.min()) / (y_prob.max() - y_prob.min())
    else:
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    roc  = roc_auc_score(y_test, y_prob)
    pr   = average_precision_score(y_test, y_prob)

    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(f'  Precision : {prec:.4f}  |  Recall : {rec:.4f}')
    print(f'  F1        : {f1:.4f}  |  ROC-AUC : {roc:.4f}')
    print(f'  PR-AUC    : {pr:.4f}')
    print(f'  TP={tp} | TN={tn} | FP={fp} | FN={fn}')
    return {'model': name, 'precision': round(prec,4), 'recall': round(rec,4),
            'f1': round(f1,4), 'roc_auc': round(roc,4), 'pr_auc': round(pr,4),
            'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn)}

results = []
results.append(eval_model('Logistic Regression', lr, X_test, y_test))
results.append(eval_model('Random Forest',       rf, X_test, y_test))
results.append(eval_model('XGBoost',             xgb, X_test, y_test))
results.append(eval_model('Isolation Forest',    iso, X_test, y_test, is_iso=True))

print('\n\n=== Model Comparison ===')
pd.DataFrame(results).set_index('model')

In [ ]:
# â”€â”€ Step 10: Save metrics + feature names for FastAPI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import json

# Save evaluation results so FastAPI can serve them
with open('models/metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved â†’ models/metrics.json')

# Save feature column names (needed for prediction)
feature_names = list(X_train.columns)
with open('models/feature_names.json', 'w') as f:
    json.dump(feature_names, f)
print('Saved â†’ models/feature_names.json')

# Save XGBoost feature importance
feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=False).head(15)
feat_imp.to_json('models/feature_importance.json', orient='records')
print('Saved â†’ models/feature_importance.json')

print('\nAll models trained and saved!')
print('Files in models/:')
for f in os.listdir('models'):
    size = os.path.getsize(f'models/{f}') / 1e6
    print(f'  {f:40s} {size:.2f} MB')

In [ ]:
# â”€â”€ Step 11: Download all models as a zip â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import shutil
from google.colab import files

# Create zip of models folder
shutil.make_archive('fraudshield_models', 'zip', 'models')
print('Created fraudshield_models.zip')

# Download it
files.download('fraudshield_models.zip')
print('\nDONE! Extract the zip into your backend/models/ folder.')
print('Then run: uvicorn app:app --reload --port 8000')

## After Downloading

1. Extract `fraudshield_models.zip`
2. Copy all files into `backend/models/`
3. Your folder should look like:
```
backend/models/
â”œâ”€â”€ logistic_regression.pkl
â”œâ”€â”€ random_forest.pkl
â”œâ”€â”€ xgboost_model.pkl
â”œâ”€â”€ isolation_forest.pkl
â”œâ”€â”€ metrics.json
â”œâ”€â”€ feature_names.json
â””â”€â”€ feature_importance.json
```
4. Start the backend â€” it will automatically load the real models:
```bash
cd backend
uvicorn app:app --reload --port 8000
```
5. The React dashboard at `http://localhost:5173` will now show real metrics!